# 00-Cleanup — Remove old test tenants from Neptune

Run this to clean stale data from previous notebook runs.


In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
gs = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()

# List all labels (tenants)
result = gs.execute_query('MATCH (n) RETURN DISTINCT labels(n) as lbl, count(n) as cnt ORDER BY cnt DESC LIMIT 50')
print('Current labels in Neptune:')
for row in result:
    print(f'  {row["lbl"][0]}: {row["cnt"]} nodes')


In [ ]:
# Define which tenants to KEEP
KEEP_TENANTS = ['hybrid_demo', 'mandela']  # active tenants

# Find old tenants to remove
old_tenants = set()
for row in result:
    label = row['lbl'][0]
    parts = label.split('__')
    if len(parts) >= 3:
        tenant = parts[2]
        if tenant not in KEEP_TENANTS:
            old_tenants.add(tenant)

print(f'Tenants to remove: {old_tenants}')
print(f'Tenants to keep: {KEEP_TENANTS}')


In [ ]:
# DELETE old tenant data (uncomment to run)
for tenant in old_tenants:
    # Delete all nodes with labels containing this tenant
    count_result = gs.execute_query(f"MATCH (n) WHERE labels(n)[0] CONTAINS '{tenant}' RETURN count(n) as cnt")
    cnt = count_result[0]['cnt'] if count_result else 0
    if cnt > 0:
        gs.execute_query(f"MATCH (n) WHERE labels(n)[0] CONTAINS '{tenant}' DETACH DELETE n")
        print(f'  Deleted {cnt} nodes from tenant: {tenant}')
    else:
        print(f'  {tenant}: already empty')

print('\nCleanup complete')


In [ ]:
# Verify — show remaining data
result = gs.execute_query('MATCH (n) RETURN DISTINCT labels(n) as lbl, count(n) as cnt ORDER BY cnt DESC LIMIT 30')
print('Remaining labels:')
for row in result:
    print(f'  {row["lbl"][0]}: {row["cnt"]}')
